# Squad and Manager Data Experiment

This notebook tests whether externally sourced pre-tournament player/squad and manager data improves the World Cup outcome model.

Source: Wikipedia squad pages for the 2010, 2014, 2018, and 2022 FIFA World Cups. These pages include squad age, caps, goals where available, position, club, and coach/manager text. The notebook downloads and parses those pages at execution time and does not write generated CSV report files.

In [1]:
from pathlib import Path
import re
import ssl
import urllib.request
import warnings

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'processed' / 'matches_with_elo.csv'
YEARS = [2010, 2014, 2018, 2022]
SEED = 20260707
LABELS = {'H': 0, 'D': 1, 'A': 2}
ORDERED = np.array(['H', 'D', 'A'])

matches = pd.read_csv(DATA, parse_dates=['date']).sort_values(['date', 'match_id']).reset_index(drop=True)
ctx = ssl._create_unverified_context()


In [2]:
TEAM_ALIASES = {
    'Bosnia and Herzegovina': 'Bosnia and Herzegovina',
    'Côte d\'Ivoire': 'Ivory Coast',
    'Ivory Coast': 'Ivory Coast',
    'Korea Republic': 'South Korea',
    'South Korea': 'South Korea',
    'IR Iran': 'Iran',
    'United States': 'United States',
    'United States of America': 'United States',
    'North Korea': 'North Korea',
    'Serbia and Montenegro': 'Serbia and Montenegro',
}

TOP_CLUB_WORDS = {
    'Manchester City', 'Real Madrid', 'Barcelona', 'Bayern Munich', 'Paris Saint-Germain', 'PSG', 'Chelsea',
    'Liverpool', 'Manchester United', 'Tottenham Hotspur', 'Arsenal', 'Juventus', 'Inter Milan', 'Milan',
    'Atlético Madrid', 'Atletico Madrid', 'Borussia Dortmund', 'Napoli', 'Roma', 'Ajax', 'Benfica', 'Porto',
}

def fetch_soup(year):
    url = f'https://en.wikipedia.org/wiki/{year}_FIFA_World_Cup_squads'
    req = urllib.request.Request(url, headers={'User-Agent': 'MatchCast research notebook; contact local user'})
    return BeautifulSoup(urllib.request.urlopen(req, context=ctx, timeout=30).read(), 'html.parser')

def parse_age(text):
    match = re.search(r'aged\s+(\d+)', text)
    return float(match.group(1)) if match else np.nan

def parse_number(text):
    match = re.search(r'-?\d+', str(text).replace(',', ''))
    return float(match.group(0)) if match else np.nan

def clean_cell(cell):
    return cell.get_text(' ', strip=True).replace('\xa0', ' ')

def nearest_team_heading(table):
    heading = table.find_previous(['h2', 'h3', 'h4'])
    if heading is None:
        return None
    text = heading.get_text(' ', strip=True)
    return re.sub(r'\[.*?\]', '', text).strip()

def parse_coach_name(soup, team):
    text = soup.get_text('\n', strip=True)
    pattern = re.compile(re.escape(team) + r'.{0,1500}?Coach:\s*([^\n]+)', re.S)
    match = pattern.search(text)
    if not match:
        return ''
    return re.sub(r'\[.*?\]', '', match.group(1)).strip()

def parse_squad_page(year):
    soup = fetch_soup(year)
    rows = []
    for table in soup.find_all('table', {'class': 'wikitable'}):
        team = nearest_team_heading(table)
        if not team or team in {'Statistics', 'Notes', 'References'}:
            continue
        header_cells = table.find_all('tr')[0].find_all(['th', 'td'])
        headers = [clean_cell(c) for c in header_cells]
        if 'Player' not in headers or 'Pos.' not in headers or 'Caps' not in headers:
            continue
        coach = parse_coach_name(soup, team)
        team_clean = TEAM_ALIASES.get(team, team)
        for tr in table.find_all('tr')[1:]:
            cells = [clean_cell(c) for c in tr.find_all(['th', 'td'])]
            if len(cells) < len(headers):
                continue
            record = dict(zip(headers, cells))
            pos_text = record.get('Pos.', record.get('Pos', ''))
            pos_match = re.search(r'(GK|DF|MF|FW)', pos_text)
            pos = pos_match.group(1) if pos_match else ''
            if pos not in {'GK', 'DF', 'MF', 'FW'}:
                continue
            club = record.get('Club', '')
            rows.append({
                'year': year,
                'team': team_clean,
                'raw_team': team,
                'coach': coach,
                'position': pos,
                'age': parse_age(record.get('Date of birth (age)', '')),
                'caps': parse_number(record.get('Caps', '')),
                'goals': parse_number(record.get('Goals', 0)),
                'club': club,
                'top_club_flag': float(any(word.lower() in club.lower() for word in TOP_CLUB_WORDS)),
            })
    return pd.DataFrame(rows)

squad_players = pd.concat([parse_squad_page(year) for year in YEARS], ignore_index=True)
display(squad_players.groupby('year').agg(players=('team', 'count'), teams=('team', 'nunique')).reset_index())
assert squad_players.groupby('year').team.nunique().min() >= 32


,year,players,teams
0,2010,736,32
1,2014,736,32
2,2018,736,32
3,2022,831,32


In [3]:
def squad_team_features(players):
    g = players.groupby(['year', 'team'])
    out = g.agg(
        squad_players=('team', 'count'),
        squad_avg_age=('age', 'mean'),
        squad_std_age=('age', 'std'),
        squad_total_caps=('caps', 'sum'),
        squad_avg_caps=('caps', 'mean'),
        squad_total_goals=('goals', 'sum'),
        squad_avg_goals=('goals', 'mean'),
        squad_top_club_players=('top_club_flag', 'sum'),
        coach=('coach', 'first'),
    ).reset_index()
    pos = players.pivot_table(index=['year', 'team'], columns='position', values='club', aggfunc='count', fill_value=0).reset_index()
    pos.columns = [str(c) for c in pos.columns]
    out = out.merge(pos, on=['year', 'team'], how='left')
    for col in ['GK', 'DF', 'MF', 'FW']:
        if col not in out:
            out[col] = 0
        out[f'squad_{col.lower()}_count'] = out[col].astype(float)
    out = out.drop(columns=[c for c in ['GK', 'DF', 'MF', 'FW'] if c in out])
    out = out.sort_values(['team', 'year']).reset_index(drop=True)
    out['coach_same_as_previous_wc'] = (out.coach.eq(out.groupby('team').coach.shift(1))).astype(float)
    out['coach_known'] = out.coach.ne('').astype(float)
    return out

team_squad = squad_team_features(squad_players)
display(team_squad.head())
display(team_squad.groupby('year').team.nunique())


,year,team,squad_players,squad_avg_age,squad_std_age,squad_total_caps,squad_avg_caps,squad_total_goals,squad_avg_goals,squad_top_club_players,coach,squad_gk_count,squad_df_count,squad_mf_count,squad_fw_count,coach_same_as_previous_wc,coach_known
0,2010,Algeria,23,26.217391,3.246768,482.0,20.956522,0.0,0.000000,0.0,Rabah Saâdane,3.0,8.0,8.0,4.0,0.0,1.0
1,2014,Algeria,23,26.173913,3.214345,381.0,16.565217,0.0,0.000000,4.0,Vahid Halilhodžić,3.0,8.0,9.0,3.0,0.0,1.0
2,2010,Argentina,23,27.130435,4.515673,562.0,24.434783,0.0,0.000000,11.0,Diego Maradona,3.0,7.0,7.0,6.0,0.0,1.0
3,2014,Argentina,23,28.521739,2.711416,741.0,32.217391,0.0,0.000000,14.0,Alejandro Sabella,3.0,7.0,8.0,5.0,0.0,1.0
4,2018,Argentina,23,29.043478,3.636660,859.0,37.347826,172.0,7.478261,13.0,Jorge Sampaoli,3.0,8.0,7.0,5.0,0.0,1.0


year
2010    32
2014    32
2018    32
2022    32
Name: team, dtype: int64

In [4]:
def classify_tournament(name):
    text = str(name).lower()
    if 'qualif' in text:
        return 'qualifier'
    if str(name) == 'FIFA World Cup':
        return 'world_cup'
    if any(marker in text for marker in ['copa america', 'copa américa', 'african cup of nations', 'afc asian cup', 'uefa euro', 'european championship', 'gold cup', 'oceania', 'nations league']):
        return 'continental'
    if text == 'friendly':
        return 'friendly'
    return 'other'

def add_basic_match_features(frame):
    out = frame.copy()
    out['year'] = out.date.dt.year
    out['tournament_type'] = out.tournament.map(classify_tournament)
    out['abs_elo_difference'] = out.elo_difference.abs()
    out['elo_similarity'] = np.exp(-out.abs_elo_difference / 200.0)
    out['elo_difference_sq'] = np.sign(out.elo_difference) * (out.elo_difference / 400.0) ** 2
    return out

def add_squad_features(frame):
    out = add_basic_match_features(frame)
    squad_cols = [c for c in team_squad.columns if c not in {'year', 'team', 'coach'}]
    home = team_squad[['year', 'team', *squad_cols]].rename(columns={'team': 'home_team', **{c: f'home_{c}' for c in squad_cols}})
    away = team_squad[['year', 'team', *squad_cols]].rename(columns={'team': 'away_team', **{c: f'away_{c}' for c in squad_cols}})
    out = out.merge(home, on=['year', 'home_team'], how='left').merge(away, on=['year', 'away_team'], how='left')
    for c in squad_cols:
        out[f'{c}_diff'] = out[f'home_{c}'] - out[f'away_{c}']
        out[f'{c}_sum'] = out[f'home_{c}'] + out[f'away_{c}']
    return out

wc = matches[(matches.tournament == 'FIFA World Cup') & matches.date.dt.year.isin(YEARS)].copy()
featured = add_squad_features(wc)
coverage = featured.groupby('year').apply(lambda g: pd.Series({
    'matches': len(g),
    'home_squad_missing': g.home_squad_players.isna().sum(),
    'away_squad_missing': g.away_squad_players.isna().sum(),
}), include_groups=False).reset_index()
display(coverage)
assert coverage[['home_squad_missing', 'away_squad_missing']].to_numpy().sum() == 0


,year,matches,home_squad_missing,away_squad_missing
0,2010,64,0,0
1,2014,64,0,0
2,2018,64,0,0
3,2022,64,0,0


In [5]:
BASE_FEATURES = ['elo_difference', 'abs_elo_difference', 'elo_difference_sq', 'elo_similarity', 'neutral']
SQUAD_FEATURES = [c for c in featured.columns if c.endswith('_diff') or c.endswith('_sum')]
TOURNAMENT_LEVELS = ['world_cup', 'qualifier', 'continental', 'friendly']

def labels_of(frame):
    return frame.result.map(LABELS).to_numpy()

def design(frame, include_squad):
    cols = BASE_FEATURES + (SQUAD_FEATURES if include_squad else [])
    out = frame[cols].astype(float).copy()
    for col in [c for c in out.columns if 'elo' in c]:
        out[col] = out[col] / 400.0
    out['neutral'] = frame.neutral.astype(float)
    return out.fillna(0.0)

def normalize(probs):
    probs = np.clip(np.asarray(probs, dtype=float), 1e-12, None)
    return probs / probs.sum(axis=1, keepdims=True)

def score(frame, probs):
    y = labels_of(frame)
    actual = np.eye(3)[y]
    return {
        'log_loss': log_loss(y, probs, labels=[0, 1, 2]),
        'brier': float(np.mean(np.sum((probs - actual) ** 2, axis=1))),
        'accuracy': float((probs.argmax(axis=1) == y).mean()),
        'correct': int((probs.argmax(axis=1) == y).sum()),
    }

rows = []
# 2010 has no previous squad-labelled tournament in this notebook, so the comparable squad model starts at 2014.
for year in [2014, 2018, 2022]:
    train = featured[featured.year < year].copy()
    test = featured[featured.year == year].copy()
    assert len(test) == 64 and len(train) >= 64
    for name, include_squad in [('elo_only_wc_squad_subset', False), ('squad_manager_model', True)]:
        model = make_pipeline(StandardScaler(), LogisticRegression(C=0.2, max_iter=3000, random_state=SEED))
        model.fit(design(train, include_squad), labels_of(train))
        probs = normalize(model.predict_proba(design(test, include_squad)))
        rows.append({'world_cup': year, 'model': name, 'matches': len(test), **score(test, probs)})

metrics = pd.DataFrame(rows)
aggregate = metrics.groupby('model').apply(lambda g: pd.Series({
    'log_loss': np.average(g.log_loss, weights=g.matches),
    'brier': np.average(g.brier, weights=g.matches),
    'accuracy': np.average(g.accuracy, weights=g.matches),
    'correct': int(g.correct.sum()),
    'matches': int(g.matches.sum()),
}), include_groups=False).sort_values(['accuracy', 'log_loss'], ascending=[False, True])

display(metrics.pivot(index='world_cup', columns='model', values='accuracy').round(4))
display(aggregate.round(4))

base = aggregate.loc['elo_only_wc_squad_subset']
squad = aggregate.loc['squad_manager_model']
print(f"Comparable WC-only Elo/formless baseline: {int(base.correct)}/{int(base.matches)} = {base.accuracy:.2%}")
print(f"Squad/manager model: {int(squad.correct)}/{int(squad.matches)} = {squad.accuracy:.2%}")
print(f"Delta: {int(squad.correct - base.correct):+d} matches, {squad.accuracy - base.accuracy:+.2%}")

assert metrics[['log_loss', 'brier', 'accuracy']].apply(np.isfinite).all().all()


model,elo_only_wc_squad_subset,squad_manager_model
world_cup,,
2014,0.5156,0.4375
2018,0.5625,0.4844
2022,0.5000,0.4531


,log_loss,brier,accuracy,correct,matches
model,,,,,
elo_only_wc_squad_subset,1.0252,0.6025,0.5260,101.0,192.0
squad_manager_model,1.2940,0.7286,0.4583,88.0,192.0


Comparable WC-only Elo/formless baseline: 101/192 = 52.60%
Squad/manager model: 88/192 = 45.83%
Delta: -13 matches, -6.77%
